# Assembly
**Megahit**
https://www.metagenomics.wiki/tools/assembly/megahit
- de novo assembly (w/o reference genome)
- aligns/assembles short reads together to reconstruct one 'metagenome'
- assembled contigs are stored in fasta file

In [22]:
# Using trimmed, qc seqs from /trimmed
# separate into groups based on metadata 
    # spp x health status x sampledata - created in reads_counts. groups found in reads_meta
# 1)remove host from sample reads
# 2)remove symbiont and human reads
# 3)concatenate all f and r seqs into single file (1 for f, 1 for r)
# 4)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, 
    #and ensures there are no gaps - larger portions of genomes if not all are now together in one sequence)

In [23]:
# based on reads_meta groups, make folders and separate samples out

## Metadata and File Setup

In [36]:
import pandas as pd
import numpy as np
import os 
from pathlib import Path

In [25]:
os.chdir("/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw")

In [31]:
reads_meta = pd.read_csv("reads_meta.csv")
reads_meta.head()

,sampleid,raw,trimmed,pct_yield,Month_year,CollectionDate,Transect,TransectNum,NewTagNum,Species,SampleNum,Health_status,colony_id,group
0,052022_BEL_CBC_T1_10_PSTR,68572842,68449647,99.82,52022.0,5/21/22,CBC30N,1.0,4,PSTR,10.0,Diseased_Margin,T1_4_PSTR,52022_PSTR_Diseased_Margin
1,052022_BEL_CBC_T1_10_PSTR,68572842,68449647,99.82,52022.0,5/21/22,CBC30N,1.0,12,PSTR,10.0,Healthy,T1_12_PSTR,52022_PSTR_Healthy
2,052022_BEL_CBC_T1_11_PSTR,48741905,48660782,99.83,52022.0,5/21/22,CBC30N,1.0,4,PSTR,11.0,Diseased_Tissue,T1_4_PSTR,52022_PSTR_Diseased_Tissue
3,052022_BEL_CBC_T1_11_PSTR,48741905,48660782,99.83,52022.0,5/21/22,CBC30N,1.0,12,PSTR,11.0,Healthy,T1_12_PSTR,52022_PSTR_Healthy
4,052022_BEL_CBC_T1_12_MCAV,166267321,165624856,99.61,52022.0,5/21/22,CBC30N,1.0,8,MCAV,12.0,Diseased_Margin,T1_8_MCAV,52022_MCAV_Diseased_Margin


In [35]:
spp_list=reads_meta['Species'].unique()
print(spp_list)

['PSTR' 'MCAV' 'PAST' 'ORBI' 'MMEA' 'NEG']


In [33]:
reads_meta['group'].unique()

array(['52022_PSTR_Diseased_Margin', '52022_PSTR_Healthy',
       '52022_PSTR_Diseased_Tissue', '52022_MCAV_Diseased_Margin',
       '52022_MCAV_Diseased_Tissue', '52022_PAST_Healthy',
       '52022_OANN_Healthy', '52022_PAST_Diseased_Tissue',
       '52022_MCAV_Healthy', '52022_OFAV_Healthy',
       '52022_PAST_Diseased_Margin', '62019_MMEA_Healthy',
       '62019_PAST_Healthy', '62019_MCAV_Healthy', '102019_PSTR_Healthy',
       '122022_OANN_Diseased_Margin', '122022_PSTR_Healthy',
       '122022_OANN_Healthy', '122022_PSTR_Diseased_Tissue',
       '122022_PSTR_Diseased_Margin', '122022_PAST_Diseased_Margin',
       '122022_OANN_Diseased_Tissue', '122022_PAST_Diseased_Tissue',
       '122022_MCAV_Diseased_Tissue', '122022_PAST_Healthy',
       '122022_MCAV_Healthy', '122022_OFAV_Diseased_Margin',
       '122022_OFAV_Diseased_Tissue', '122022_OFAV_Healthy',
       '122022_MCAV_Diseased_Margin', 'Negative'], dtype=object)

In [ ]:
# make sample list for each spp and group? 

In [ ]:
# common variables to use in scripts
BASE_DIR = "/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw"
STOREREADS = "reads_filtered.txt" # read_count,step,sampleid

In [ ]:
# separating into diff steps 

## SBATCH SCRIPTS

### Coral Host Removal - by Spp

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long
#SBATCH -t 168:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-assembly-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw

# set paths for existing bowtie genome indices
MCAV_index=Mcav_DB
MCAV_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Mcav_genome/"
MMEA_index=Mmea_DB
MMEA_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Mmea_genome/"
ORBI_index=Ofav_DB
ORBI_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Ofav_genome/"
# use ssid for past (no PAST host genome - closest relative for genomes we have)
PAST_index=Ssid_DB
PAST_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Ssid_genome/"
# use cnat for pstr (no PSTR host genome- closest relative for genomes we have)
PSTR_index=Cnat_DB
PSTR_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Cnat_genome/"
 
# get unique species from list of samples, spp, and groups
spp_list=$(cut -f 2 filtered_sample_groups.txt | tail -n +2 | sort -u)
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"

# loop through spp list...
for spp in $spp_list; do
    # make spp folder if it doesn't already exist 
    mkdir -p "$spp"

    # identify samples for each spp to host remove together 
    samples=$(awk -F'\t' -v s="$spp" '$2 == s {print $1}' filtered_sample_groups.txt)

    # copy samples to spp folders
    for id in $samples; do
        if [[ -f "${READSPATH}/${id}_R1_001_val_1.fq" ]] && [[ -f "${READSPATH}/${id}_R2_001_val_2.fq" ]]; then
            # cp "$READSPATH/${id}_R1_001_val_1.fq" "$spp/"
            # cp "$READSPATH/${id}_R2_001_val_2.fq" "$spp/"
            echo "all ${id} files present in $spp"
        else
            echo "Missing files for $spp: $id in $READSPATH"
        fi
    done

    # create file with samplelist and groups for each spp 
    (awk -F'\t' -v s="$spp" '$2 == s' filtered_sample_groups.txt) | cut -f1,3- > $spp/spp_samples 

# 1)remove host from sample reads
# Host seq removal - Thij's script https://github.com/ThijsSt/SCTLD-metagenomes/blob/main/Quality_control_metagenomes.ipynb
    # by specie
    FINALREADS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
    WORKINGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed/temp"
    mkdir -p $FINALREADS
    mkdir -p $WORKINGPATH

    # assigning path and index variable for each spp       
    spp_index="${spp}_index"
    spp_path="${spp}_path"        
    input_index="${!spp_index}"
    input_path="${!spp_path}"
    
    #skip bowtie index build - already done
    #loop through samples in each spp group
    for id in $samples; do
        #re-align reads back to the index (host genome)
        bowtie2 -p 8 -x $input_path/$input_index -1 $READSPATH/"${id}_R1_001_val_1.fq" -2 $READSPATH/"${id}_R2_001_val_2.fq" -S $WORKINGPATH/"${id}"_mapped_and_unmapped.sam
        
        #convert sam file from bowtie to a bam file for processing
        samtools view -bS $WORKINGPATH/"${id}"_mapped_and_unmapped.sam > $WORKINGPATH/"${id}"_mapped_and_unmapped.bam
        
        #extract only the reads of which both do not match against the host genome
        samtools view -b -f 12 -F 256 $WORKINGPATH/"${id}"_mapped_and_unmapped.bam > $WORKINGPATH/"${id}"_bothReadsUnmapped.bam
        
        # sorts the file so both mates are together and then extracts them back as .fastq files
        samtools sort -n -m 5G -@ 2 $WORKINGPATH/"${id}"_bothReadsUnmapped.bam -o $WORKINGPATH/"${id}"_bothReadsUnmapped_sorted.bam
        samtools fastq -@ 8 $WORKINGPATH/"${id}"_bothReadsUnmapped_sorted.bam \
            -1 $FINALREADS/"${id}"_host_removed_R1.fastq \
            -2 $FINALREADS/"${id}"_host_removed_R2.fastq \
            -0 /dev/null -s /dev/null -n
         if [ $? -eq 0 ]; then
            echo "host removal completed successfully for sample: ${id}"
        else
            echo "host removal encountered an error for sample: ${id}"
            exit 1  
        fi
    done     
done
conda deactivate
echo "Host removal: All samples processed successfully."

# JOB-ID: 53514255
# bash script file name: host_removal

In [ ]:
# run multiple scripts for multiple spp at the same time to make faster 
# start with orbi for second script

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-assemblyORBI-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw

# set paths for spp & existing bowtie genome indices
spp="ORBI" 
ORBI_index=Ofav_DB
ORBI_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Ofav_genome/"

# identify samples for each spp to host remove together 
samples=$(awk -F'\t' -v s="$spp" '$2 == s {print $1}' filtered_sample_groups.txt)

# already checked that all samples within the spp are present before proceeding - see old versions on git
# file with samplelist and groups for each spp has already been created

# 1)remove host from sample reads
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"
FINALREADS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
WORKINGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed/temp"
mkdir -p $FINALREADS
mkdir -p $WORKINGPATH

# assigning path and index variable for each spp       
spp_index="${spp}_index"
spp_path="${spp}_path"        
input_index="${!spp_index}"
input_path="${!spp_path}"
    
#loop through samples in each spp (#skip bowtie index build - already done)
for id in $samples; do
    # check if sample has already been completed
    if [[ -f "${FINALREADS}/${id}_host_removed_R1.fastq" ]] && [[ -f "${FINALREADS}/${id}_host_removed_R2.fastq" ]]; then
        echo "${id} already completed"
    else
        # proceed with bowtie if it hasnt been completed
        bowtie2 -p 16 -x $input_path/$input_index \
            -1 "$READSPATH/${id}_R1_001_val_1.fq" \
            -2 "$READSPATH/${id}_R2_001_val_2.fq" | \
            samtools view -@ 6 -b -f 12 -F 256 - > "$WORKINGPATH/${id}_bothReadsUnmapped.bam"
        
        # sorts the file so both mates are together and then extracts them back as .fastq files
        samtools sort -n -m 4G -@ 12 "$WORKINGPATH/${id}_bothReadsUnmapped.bam" -o "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam"
        samtools fastq -@ 16 "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam" \
            -1 >(bgzip -c > "$FINALREADS/${id}_host_removed_R1.fastq.gz") \
            -2 >(bgzip -c > "$FINALREADS/${id}_host_removed_R2.fastq.gz") \
            -0 /dev/null -s /dev/null -n
         if [ $? -eq 0 ]; then
            echo "host removal completed successfully for sample: ${id}"
            # delete intermediate bam files
            rm -f "$WORKINGPATH/${id}"_*.bam
        else
            echo "host removal encountered an error for sample: ${id}"
            exit 1  
        fi
    fi
done  

conda deactivate
echo "Host removal: All samples processed successfully."

# JOB-ID: 53707659,53723284
# bash script file name: host_removal_orbi

In [ ]:
# repeat PAST

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-assemblyPAST-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw

# stnd variables
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"
spp="PAST" 
PAST_index=Past_DB
PAST_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Past_genome"

# identify samples for each spp to host remove together 
samples=$(awk -F'\t' -v s="$spp" '$2 == s {print $1}' filtered_sample_groups.txt)

# build host genome index 
bowtie2-build --threads 8 "$PAST_path/past_filtered_assembly.fasta" "$PAST_path/$PAST_index"

# already checked that all samples within the spp are present before proceeding - see old versions on git
# file with samplelist and groups for each spp has already been created

# 1)remove host from sample reads
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"
FINALREADS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed_past"
WORKINGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed_past/temp"
mkdir -p $FINALREADS
mkdir -p $WORKINGPATH

# assigning path and index variable for each spp       
spp_index="${spp}_index"
spp_path="${spp}_path"        
input_index="${!spp_index}"
input_path="${!spp_path}"
    
#loop through samples in each spp (#skip bowtie index build - already done)
for id in $samples; do
    # proceed with bowtie if it hasnt been completed
    bowtie2 -p 16 -x $input_path/$input_index \
        -1 "$READSPATH/${id}_R1_001_val_1.fq" \
        -2 "$READSPATH/${id}_R2_001_val_2.fq" | \
        samtools view -@ 6 -b -f 12 -F 256 - > "$WORKINGPATH/${id}_bothReadsUnmapped.bam"
    
    # sorts the file so both mates are together and then extracts them back as .fastq files
    samtools sort -n -m 4G -@ 12 "$WORKINGPATH/${id}_bothReadsUnmapped.bam" -o "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam"
    samtools fastq -@ 16 "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam" \
        -1 >(bgzip -c > "$FINALREADS/${id}_host_removed_R1.fastq.gz") \
        -2 >(bgzip -c > "$FINALREADS/${id}_host_removed_R2.fastq.gz") \
        -0 /dev/null -s /dev/null -n
     if [ $? -eq 0 ]; then
        echo "host removal completed successfully for sample: ${id}"
        # delete intermediate bam files
        rm -f "$WORKINGPATH/${id}"_*.bam
    else
        echo "host removal encountered an error for sample: ${id}"
        exit 1  
        fi
done  

conda deactivate
echo "Host removal: All samples processed successfully."

# JOB-ID: using ssid genome: 53707344, 53723373
#         using past genome: 
# bash script file name: host_removal_past

In [ ]:
# repeat pstr

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-assemblyPAST-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw

# stnd variables
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"

# set paths for spp & existing bowtie genome indices
spp="PSTR" 
# use cnat for pstr (no PSTR host genome- closest relative for genomes we have)
PSTR_index=Cnat_DB
PSTR_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Cnat_genome/"

# identify samples for each spp to host remove together 
samples=$(awk -F'\t' -v s="$spp" '$2 == s {print $1}' filtered_sample_groups.txt)

# already checked that all samples within the spp are present before proceeding - see old versions on git
# file with samplelist and groups for each spp has already been created

# 1)remove host from sample reads
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"
FINALREADS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
WORKINGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed/temp"
mkdir -p $FINALREADS
mkdir -p $WORKINGPATH

# assigning path and index variable for each spp       
spp_index="${spp}_index"
spp_path="${spp}_path"        
input_index="${!spp_index}"
input_path="${!spp_path}"
    
#loop through samples in each spp (#skip bowtie index build - already done)
for id in $samples; do
    # check if sample has already been completed
    if [[ -f "${FINALREADS}/${id}_host_removed_R1.fastq" ]] && [[ -f "${FINALREADS}/${id}_host_removed_R2.fastq" ]]; then
        echo "${id} already completed"
    else
        # proceed with bowtie if it hasnt been completed
        bowtie2 -p 16 -x $input_path/$input_index \
            -1 "$READSPATH/${id}_R1_001_val_1.fq" \
            -2 "$READSPATH/${id}_R2_001_val_2.fq" | \
            samtools view -@ 6 -b -f 12 -F 256 - > "$WORKINGPATH/${id}_bothReadsUnmapped.bam"
        
        # sorts the file so both mates are together and then extracts them back as .fastq files
        samtools sort -n -m 4G -@ 12 "$WORKINGPATH/${id}_bothReadsUnmapped.bam" -o "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam"
        samtools fastq -@ 16 "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam" \
            -1 >(bgzip -c > "$FINALREADS/${id}_host_removed_R1.fastq.gz") \
            -2 >(bgzip -c > "$FINALREADS/${id}_host_removed_R2.fastq.gz") \
            -0 /dev/null -s /dev/null -n
         if [ $? -eq 0 ]; then
            echo "host removal completed successfully for sample: ${id}"
            # delete intermediate bam files
            rm -f "$WORKINGPATH/${id}"_*.bam
        else
            echo "host removal encountered an error for sample: ${id}"
            exit 1  
        fi
    fi
done  

conda deactivate
echo "Host removal: All samples processed successfully."

# JOB-ID: 53707348, 53723406
# bash script file name: host_removal_pstr

In [ ]:
# single pstr sample results
# bowtie alignment
37358218 reads; of these:
  37358218 (100.00%) were paired; of these:
    23031981 (61.65%) aligned concordantly 0 times
    8128327 (21.76%) aligned concordantly exactly 1 time
    6197910 (16.59%) aligned concordantly >1 times
    ----
    23031981 pairs aligned concordantly 0 times; of these:
      66050 (0.29%) aligned discordantly 1 time
    ----
    22965931 pairs aligned 0 times concordantly or discordantly; of these:
      45931862 mates make up the pairs; of these:
        44266005 (96.37%) aligned 0 times
        1079778 (2.35%) aligned exactly 1 time
        586079 (1.28%) aligned >1 times
40.75% overall alignment rate

# samtools
[M::bam2fq_mainloop] discarded 0 singletons
[M::bam2fq_mainloop] processed 42752048 reads


# repair
java -ea -Xmx13352m -cp /home/brooke_sienkiewicz_student_uml_edu/.conda/envs/assembly/opt/bbmap-39.01-1/current/ jgi.SplitPairsAndSingles rp in1=/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/PSTR/assembly/sym_human_removed/102019_BEL_CBC_T1_29_PSTR_host_removed_R1.tagged_filter.fastq.gz in2=/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/PSTR/assembly/sym_human_removed/102019_BEL_CBC_T1_29_PSTR_host_removed_R2.tagged_filter.fastq.gz out1=/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/PSTR/assembly/final_filtered/102019_BEL_CBC_T1_29_PSTR_host_removed_R1.tagged_filter_ready.fastq.gz out2=/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/PSTR/assembly/final_filtered/102019_BEL_CBC_T1_29_PSTR_host_removed_R2.tagged_filter_ready.fastq.gz outs=/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/PSTR/assembly/final_filtered/102019_BEL_CBC_T1_29_PSTRsingletons.fq repair
Executing jgi.SplitPairsAndSingles [rp, in1=/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/PSTR/assembly/sym_human_removed/102019_BEL_CBC_T1_29_PSTR_host_removed_R1.tagged_filter.fastq.gz, in2=/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/PSTR/assembly/sym_human_removed/102019_BEL_CBC_T1_29_PSTR_host_removed_R2.tagged_filter.fastq.gz, out1=/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/PSTR/assembly/final_filtered/102019_BEL_CBC_T1_29_PSTR_host_removed_R1.tagged_filter_ready.fastq.gz, out2=/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/PSTR/assembly/final_filtered/102019_BEL_CBC_T1_29_PSTR_host_removed_R2.tagged_filter_ready.fastq.gz, outs=/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/PSTR/assembly/final_filtered/102019_BEL_CBC_T1_29_PSTRsingletons.fq, repair]

Set INTERLEAVED to false
Started output stream.

Input:                          41109018 reads          4684417468 bases.
Result:                         41109018 reads (100.00%)        4684417468 bases (100.00%)
Pairs:                          40803698 reads (99.26%)         4646282067 bases (99.19%)
Singletons:                     305320 reads (0.74%)    38135401 bases (0.81%)

Time:                           92.884 seconds.
Reads Processed:      41109k    442.58k reads/sec
Bases Processed:       4684m    50.43m bases/sec
repair completed successfully for sample: 102019_BEL_CBC_T1_29_PSTR

In [ ]:
# rerun mmea bc original script failed in the middle due to time clock

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-assemblyPAST-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw

# stnd variables
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"

# set paths for spp & existing bowtie genome indices
spp="MMEA" 
MMEA_index=Mmea_DB
MMEA_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Mmea_genome/"

# identify samples for each spp to host remove together 
samples=$(awk -F'\t' -v s="$spp" '$2 == s {print $1}' filtered_sample_groups.txt)

# already checked that all samples within the spp are present before proceeding - see old versions on git
# file with samplelist and groups for each spp has already been created

# 1)remove host from sample reads
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed"
FINALREADS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
WORKINGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed/temp"
mkdir -p $FINALREADS
mkdir -p $WORKINGPATH

# assigning path and index variable for each spp       
spp_index="${spp}_index"
spp_path="${spp}_path"        
input_index="${!spp_index}"
input_path="${!spp_path}"
    
#loop through samples in each spp (#skip bowtie index build - already done)
for id in $samples; do
    # check if sample has already been completed
    if [[ -f "${FINALREADS}/${id}_host_removed_R1.fastq" ]] && [[ -f "${FINALREADS}/${id}_host_removed_R2.fastq" ]]; then
        echo "${id} already completed"
    else
        # proceed with bowtie if it hasnt been completed
        bowtie2 -p 16 -x $input_path/$input_index \
            -1 "$READSPATH/${id}_R1_001_val_1.fq" \
            -2 "$READSPATH/${id}_R2_001_val_2.fq" | \
            samtools view -@ 6 -b -f 12 -F 256 - > "$WORKINGPATH/${id}_bothReadsUnmapped.bam"
        
        # sorts the file so both mates are together and then extracts them back as .fastq files
        samtools sort -n -m 4G -@ 12 "$WORKINGPATH/${id}_bothReadsUnmapped.bam" -o "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam"
        samtools fastq -@ 16 "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam" \
            -1 >(bgzip -c > "$FINALREADS/${id}_host_removed_R1.fastq.gz") \
            -2 >(bgzip -c > "$FINALREADS/${id}_host_removed_R2.fastq.gz") \
            -0 /dev/null -s /dev/null -n
         if [ $? -eq 0 ]; then
            echo "host removal completed successfully for sample: ${id}"
            # delete intermediate bam files
            rm -f "$WORKINGPATH/${id}"_*.bam
        else
            echo "host removal encountered an error for sample: ${id}"
            exit 1  
        fi
    fi
done  

conda deactivate
echo "Host removal: All samples processed successfully."

# JOB-ID: 53723424
# bash script file name: host_removal_mmea

In [ ]:
# will run read count scripts separately 

### Symbiont Removal and Assembly - by Spp

In [ ]:
# remove symbionts, finish assembly step 
# run for each spp, and loop through groups within spp for assembly step

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long
#SBATCH -t 96:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-mcav_assembly2-%j.out  # %j = job ID

# 2)remove symbiont and human seqs using fastq screen 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw
module load Bowtie2/2.4.5-GCC-11.3.0
module load conda/latest
#conda activate fastq_screen
FASTQSCREEN='/home/brooke_sienkiewicz_student_uml_edu/.conda/envs/fastq_screen/share/fastq-screen-0.15.3-0'

spp="MCAV"
OUTPUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
SAMPLEFILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/spp_samples"

mkdir -p "$OUTPUTDIR"
--nohits = output reads do not map to any genomes - removes human and symbiont seq matches
while IFS=$'\t' read -r SAMPLEID GROUP; do
   $FASTQSCREEN/fastq_screen --nohits --aligner bowtie2 --conf $FASTQSCREEN/fastq_screen.conf --outdir $OUTPUTDIR \
   $READSPATH/"${SAMPLEID}"_host_removed_R1.fastq $READSPATH/"${SAMPLEID}"_host_removed_R2.fastq;
    if [ $? -eq 0 ]; then
           echo "fastq_screen completed successfully for sample: $SAMPLEID"
       else
           echo "fastq_screen encountered an error for sample: $SAMPLEID"
           exit 1
       fi
done < $SAMPLEFILE
conda deactivate
echo "Symbiont, human removal: All samples processed successfully."

#From Nikea: Using repair.sh script from:https://jgi.doe.gov/data-and-tools/software-tools/bbtools/bb-tools-user-guide/repair-guide/
# re-pair scripts after fastqscreen - sometimes paired reads get unpaired during the symbiont removal phase
# make sure to install first in conda assembly env
conda activate assembly
# conda install -c bioconda bbmap

READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/final_filtered"
cd "$READSPATH"
# gzip *.fastq

while IFS=$'\t' read -r SAMPLEID GROUP; do
    if [[ -f "${OUTDIR}/${SAMPLEID}_host_removed_R1.tagged_filter_ready.fastq.gz" ]] && [[ -f "${OUTDIR}/${SAMPLEID}_host_removed_R2.tagged_filter_ready.fastq.gz" ]]; then
        echo "${SAMPLEID}: repair already completed"
    else
        repair.sh in1=$READSPATH/"${SAMPLEID}"_host_removed_R1.tagged_filter.fastq.gz in2=$READSPATH/"${SAMPLEID}"_host_removed_R2.tagged_filter.fastq.gz \
        out1=${OUTDIR}/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz out2=${OUTDIR}/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz \
        outs=${OUTDIR}/"${SAMPLEID}"singletons.fq repair;
         if [ $? -eq 0 ]; then
            echo "repair completed successfully for sample: $SAMPLEID"
        else
            echo "repair encountered an error for sample: $SAMPLEID"
            exit 1
        fi
    fi
done < "$SAMPLEFILE"
echo "Repair: All samples processed successfully."

# 2)concatenate all f and r seqs into single file (1 for f, 1 for r)
    # by group
# redo and use repaired
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/final_filtered"
spp_groups=$(cut -f 2 $SAMPLEFILE | sort -u)

for group in $spp_groups; do
    echo "Processing samples for $group"
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    rm -rf "$OUTDIR"    # delete and redo 
    mkdir -p "$OUTDIR"
# make sample list for each group and check if all files exist before concatenating
    samples=$(grep -w "$group" "$SAMPLEFILE" | cut -f 1)
    for s in $samples; do
        [ -f "$READSPATH/${s}_host_removed_R1.tagged_filter_ready.fastq.gz" ] || { echo "Missing R1 for $s"; exit 1; }
        [ -f "$READSPATH/${s}_host_removed_R2.tagged_filter_ready.fastq.gz" ] || { echo "Missing R2 for $s"; exit 1; }
    done

    # concat F & R within group if all files are present 
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R1.tagged_filter_ready.fastq.gz|" | xargs cat  > "$OUTDIR/${group}_reads_R1_ALL.fastq.gz"
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R2.tagged_filter_ready.fastq.gz|" | xargs cat  > "$OUTDIR/${group}_reads_R2_ALL.fastq.gz"
done
conda deactivate 

# 3)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, and ensures there are no gaps
conda activate assembly
for group in $spp_groups; do
    READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    echo "Processing $group"
    MEG_OUT="${READSPATH}/megahit_host_removed"
    rm -rf "$MEG_OUT"

    megahit --presets meta-large \
    -t 20 \
    -1 "${READSPATH}/${group}_reads_R1_ALL.fastq.gz" \
    -2 "${READSPATH}/${group}_reads_R2_ALL.fastq.gz" \
    -o "$MEG_OUT" \
    --out-prefix "$group"
    
    if [ $? -eq 0 ]; then
            echo "megahit completed successfully for $group"
        else
            echo "megahit encountered an error for $group"
            exit 1
        fi
done
# megahit has to make the directory; will fail if it already exists
# use

# JOB-ID: 53923368, 54683562, 54750736, 55227129
# bash script file name: mcav_assembly2

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=250G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long
#SBATCH -t 144:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-mmea_assembly2-%j.out  # %j = job ID

# 2)remove symbiont and human seqs using fastq screen 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw
module load bowtie2/2.5.2
module load conda/latest
# conda activate fastq_screen
# FASTQSCREEN='/home/brooke_sienkiewicz_student_uml_edu/.conda/envs/fastq_screen/share/fastq-screen-0.15.3-0'

spp="MMEA"
SAMPLEFILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/spp_samples"
# OUTPUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
# READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"

# mkdir -p "$OUTPUTDIR"
# # --nohits = output reads do not map to any genomes - removes human and symbiont seq matches
# while IFS=$'\t' read -r SAMPLEID GROUP; do
#     $FASTQSCREEN/fastq_screen --nohits --aligner bowtie2 --conf $FASTQSCREEN/fastq_screen.conf --outdir $OUTPUTDIR \
#     $READSPATH/"${SAMPLEID}"_host_removed_R1.fastq.gz $READSPATH/"${SAMPLEID}"_host_removed_R2.fastq.gz;
#      if [ $? -eq 0 ]; then
#             echo "fastq_screen completed successfully for sample: $SAMPLEID"
#         else
#             echo "fastq_screen encountered an error for sample: $SAMPLEID"
#             exit 1
#         fi
# done < $SAMPLEFILE
# conda deactivate
# echo "Symbiont, human removal: All samples processed successfully."
conda activate assembly

READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/final_filtered"
cd "$READSPATH"
# gzip *.fastq

while IFS=$'\t' read -r SAMPLEID GROUP; do
    if [[ -f "${OUTDIR}/${SAMPLEID}_host_removed_R1.tagged_filter_ready.fastq.gz" ]] && [[ -f "${OUTDIR}/${SAMPLEID}_host_removed_R2.tagged_filter_ready.fastq.gz" ]]; then
        echo "${SAMPLEID}: repair already completed"
    else
        repair.sh in1=$READSPATH/"${SAMPLEID}"_host_removed_R1.tagged_filter.fastq.gz in2=$READSPATH/"${SAMPLEID}"_host_removed_R2.tagged_filter.fastq.gz \
        out1=${OUTDIR}/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz out2=${OUTDIR}/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz \
        outs=${OUTDIR}/"${SAMPLEID}"singletons.fq repair;
         if [ $? -eq 0 ]; then
            echo "repair completed successfully for sample: $SAMPLEID"
        else
            echo "repair encountered an error for sample: $SAMPLEID"
            exit 1
        fi
    fi
done < "$SAMPLEFILE"
echo "Repair: All samples processed successfully."


# 2)concatenate all f and r seqs into single file (1 for f, 1 for r)
    # by group 
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/final_filtered"
spp_groups=$(cut -f 2 $SAMPLEFILE | sort -u)

for group in $spp_groups; do
    echo "Processing samples for $group"
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    mkdir -p "$OUTDIR"
# make sample list for each group and check if all files exist before concatenating
    samples=$(grep -w "$group" "$SAMPLEFILE" | cut -f 1)
    for s in $samples; do
        [ -f "$READSPATH/${s}_host_removed_R1.tagged_filter_ready.fastq.gz" ] || { echo "Missing R1 for $s"; exit 1; }
        [ -f "$READSPATH/${s}_host_removed_R2.tagged_filter_ready.fastq.gz" ] || { echo "Missing R2 for $s"; exit 1; }
    done

    # concat F & R within group if all files are present 
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R1.tagged_filter_ready.fastq.gz|" | xargs cat  > "$OUTDIR/${group}_reads_R1_ALL.fastq.gz"
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R2.tagged_filter_ready.fastq.gz|" | xargs cat  > "$OUTDIR/${group}_reads_R2_ALL.fastq.gz"
done 

# 3)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, and ensures there are no gaps
for group in $spp_groups; do
    READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    echo "Processing $group"
    MEG_OUT="${READSPATH}/megahit_host_removed"

    megahit --presets meta-large \
    -t 20 \
    -1 "${READSPATH}/${group}_reads_R1_ALL.fastq.gz" \
    -2 "${READSPATH}/${group}_reads_R2_ALL.fastq.gz" \
    -o "$MEG_OUT" \
    --out-prefix "$group"
    if [ $? -eq 0 ]; then
            echo "megahit completed successfully for $group"
        else
            echo "megahit encountered an error for $group"
            exit 1
        fi
done
# megahit has to make the directory; will fail if it already exists
conda deactivate

# JOB-ID: 54684038, 55240769, 55305439
# bash script file name: mmea_assembly2

In [ ]:
## ORBI

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=250G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long
#SBATCH -t 144:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-orbi_assembly2-%j.out  # %j = job ID

# 2)remove symbiont and human seqs using fastq screen 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw
module load bowtie2/2.5.2
module load conda/latest
# conda activate fastq_screen
# FASTQSCREEN='/home/brooke_sienkiewicz_student_uml_edu/.conda/envs/fastq_screen/share/fastq-screen-0.15.3-0'

spp="ORBI"
OUTPUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
SAMPLEFILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/spp_samples"

# mkdir -p "$OUTPUTDIR"
# # zip any unzipped host_removed files - idk what happened in previous step 
# cd "$READSPATH"
# gzip *.fastq
# # --nohits = output reads do not map to any genomes - removes human and symbiont seq matches
# while IFS=$'\t' read -r SAMPLEID GROUP; do
#     $FASTQSCREEN/fastq_screen --nohits --aligner bowtie2 --conf $FASTQSCREEN/fastq_screen.conf --outdir $OUTPUTDIR \
#     $READSPATH/"${SAMPLEID}"_host_removed_R1.fastq.gz $READSPATH/"${SAMPLEID}"_host_removed_R2.fastq.gz;
#      if [ $? -eq 0 ]; then
#             echo "fastq_screen completed successfully for sample: $SAMPLEID"
#         else
#             echo "fastq_screen encountered an error for sample: $SAMPLEID"
#             exit 1
#         fi
# done < "$SAMPLEFILE"
# conda deactivate
# echo "Symbiont, human removal: All samples processed successfully."
conda activate assembly 
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/final_filtered"
cd "$READSPATH"

while IFS=$'\t' read -r SAMPLEID GROUP; do
    if [[ -f "${OUTDIR}/${SAMPLEID}_host_removed_R1.tagged_filter_ready.fastq.gz" ]] && [[ -f "${OUTDIR}/${SAMPLEID}_host_removed_R2.tagged_filter_ready.fastq.gz" ]]; then
        echo "${SAMPLEID}: repair already completed"
    else
        repair.sh in1=$READSPATH/"${SAMPLEID}"_host_removed_R1.tagged_filter.fastq.gz in2=$READSPATH/"${SAMPLEID}"_host_removed_R2.tagged_filter.fastq.gz \
        out1=${OUTDIR}/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz out2=${OUTDIR}/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz \
        outs=${OUTDIR}/"${SAMPLEID}"singletons.fq repair;
         if [ $? -eq 0 ]; then
            echo "repair completed successfully for sample: $SAMPLEID"
        else
            echo "repair encountered an error for sample: $SAMPLEID"
            exit 1
        fi
    fi
done < "$SAMPLEFILE"
echo "Repair: All samples processed successfully."


# 2)concatenate all f and r seqs into single file (1 for f, 1 for r)
    # by group
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/final_filtered"
spp_groups=$(cut -f 2 $SAMPLEFILE | sort -u)

for group in $spp_groups; do
    echo "Processing samples for $group"
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    rm -rf "$OUTDIR"    # delete and redo     
    mkdir -p "$OUTDIR"
# make sample list for each group and check if all files exist before concatenating
    samples=$(grep -w "$group" "$SAMPLEFILE" | cut -f 1)
    for s in $samples; do
        [ -f "$READSPATH/${s}_host_removed_R1.tagged_filter_ready.fastq.gz" ] || { echo "Missing R1 for $s"; exit 1; }
        [ -f "$READSPATH/${s}_host_removed_R2.tagged_filter_ready.fastq.gz" ] || { echo "Missing R2 for $s"; exit 1; }
    done

    # concat F & R within group if all files are present 
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R1.tagged_filter_ready.fastq.gz|" | xargs cat > "$OUTDIR/${group}_reads_R1_ALL.fastq.gz"
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R2.tagged_filter_ready.fastq.gz|" | xargs cat > "$OUTDIR/${group}_reads_R2_ALL.fastq.gz"
done

# 3)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, and ensures there are no gaps
for group in $spp_groups; do
    READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    echo "Processing $group"
    MEG_OUT="${READSPATH}/megahit_host_removed"
    rm -rf "$MEG_OUT"

    megahit --presets meta-large \
    -t 20 \
    -1 "${READSPATH}/${group}_reads_R1_ALL.fastq.gz" \
    -2 "${READSPATH}/${group}_reads_R2_ALL.fastq.gz" \
    -o "$MEG_OUT" \
    --out-prefix "$group"
    
    if [ $? -eq 0 ]; then
            echo "megahit completed successfully for $group"
        else
            echo "megahit encountered an error for $group"
            exit 1
        fi
done
# megahit has to make the directory; will fail if it already exists
conda deactivate

# JOB-ID: 53971713, 54693815, 55241040, 55305507
# bash script file name: orbi_assembly2

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=250G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long
#SBATCH -t 144:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/PAST/slurm-PAST_assembly2-%j.out  # %j = job ID

# 2)remove symbiont and human seqs using fastq screen 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw
module load bowtie2/2.5.2
module load conda/latest
conda activate fastq_screen
FASTQSCREEN='/home/brooke_sienkiewicz_student_uml_edu/.conda/envs/fastq_screen/share/fastq-screen-0.15.3-0'

spp="PAST"
OUTPUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed2"
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed_past"
SAMPLEFILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/spp_samples"

mkdir -p "$OUTPUTDIR"
# cd "$READSPATH"
# gzip *.fastq

# --nohits = output reads do not map to any genomes - removes human and symbiont seq matches
while IFS=$'\t' read -r SAMPLEID GROUP; do
    # check if file exists first since first script run failed halfway through
    # CHECK_FILE="${OUTPUTDIR}/${SAMPLEID}_host_removed_R1.tagged_filter.fastq.gz"
    # if [ -f "$CHECK_FILE" ]; then
    #     echo "Sample $SAMPLEID already processed. Skipping..."
    #     continue
    # fi
    
    # echo "Processing sample: $SAMPLEID"
    $FASTQSCREEN/fastq_screen --nohits --aligner bowtie2 --conf $FASTQSCREEN/fastq_screen.conf --outdir $OUTPUTDIR \
    $READSPATH/"${SAMPLEID}"_host_removed_R1.fastq.gz $READSPATH/"${SAMPLEID}"_host_removed_R2.fastq.gz;
    if [ $? -eq 0 ]; then
            echo "fastq_screen completed successfully for sample: $SAMPLEID"
    else
            echo "fastq_screen encountered an error for sample: $SAMPLEID"
            exit 1
    fi
done < "$SAMPLEFILE"
conda deactivate
echo "Symbiont, human removal: All samples processed successfully."

conda activate assembly
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed2"
OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/final_filtered2"
cd "$READSPATH"

while IFS=$'\t' read -r SAMPLEID GROUP; do
    # if [[ -f "${OUTDIR}/${SAMPLEID}_host_removed_R1.tagged_filter_ready.fastq.gz" ]] && [[ -f "${OUTDIR}/${SAMPLEID}_host_removed_R2.tagged_filter_ready.fastq.gz" ]]; then
    #     echo "${SAMPLEID}: repair already completed"
    # else
    repair.sh in1=$READSPATH/"${SAMPLEID}"_host_removed_R1.tagged_filter.fastq.gz in2=$READSPATH/"${SAMPLEID}"_host_removed_R2.tagged_filter.fastq.gz \
    out1=${OUTDIR}/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz out2=${OUTDIR}/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz \
    outs=${OUTDIR}/"${SAMPLEID}"singletons.fq repair;
     if [ $? -eq 0 ]; then
        echo "repair completed successfully for sample: $SAMPLEID"
    else
        echo "repair encountered an error for sample: $SAMPLEID"
        exit 1
    fi
    # fi
done < "$SAMPLEFILE"
echo "Repair: All samples processed successfully."


# 2)concatenate all f and r seqs into single file (1 for f, 1 for r)
    # by group
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/final_filtered2"
spp_groups=$(cut -f 2 $SAMPLEFILE | sort -u)

for group in $spp_groups; do
    echo "Processing samples for $group"
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/megahit_redo/${group}"
    mkdir -p "$OUTDIR"
# make sample list for each group and check if all files exist before concatenating
    samples=$(grep -w "$group" "$SAMPLEFILE" | cut -f 1)
    for s in $samples; do
        [ -f "$READSPATH/${s}_host_removed_R1.tagged_filter_ready.fastq.gz" ] || { echo "Missing R1 for $s"; exit 1; }
        [ -f "$READSPATH/${s}_host_removed_R2.tagged_filter_ready.fastq.gz" ] || { echo "Missing R2 for $s"; exit 1; }
    done

    # concat F & R within group if all files are present 
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R1.tagged_filter_ready.fastq.gz|" | xargs cat > "$OUTDIR/${group}_reads_R1_ALL.fastq.gz"
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R2.tagged_filter_ready.fastq.gz|" | xargs cat > "$OUTDIR/${group}_reads_R2_ALL.fastq.gz"
done

# 3)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, and ensures there are no gaps
for group in $spp_groups; do
    READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/megahit_redo/${group}"
    echo "Processing $group"
    MEG_OUT="${READSPATH}/megahit_host_removed"

    megahit --presets meta-large \
    -t 20 \
    -1 "${READSPATH}/${group}_reads_R1_ALL.fastq.gz" \
    -2 "${READSPATH}/${group}_reads_R2_ALL.fastq.gz" \
    -o "$MEG_OUT" \
    --out-prefix "$group"
    
    if [ $? -eq 0 ]; then
            echo "megahit completed successfully for $group"
        else
            echo "megahit encountered an error for $group"
            exit 1
        fi
done
# megahit has to make the directory; will fail if it already exists

# JOB-ID: 53972006, 54694069, 55229678, 55328445, 55515152, 55571270
# bash script file name: PAST_assembly2

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=250G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long
#SBATCH -t 144:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-PSTR_assembly2-%j.out  # %j = job ID

# 2)remove symbiont and human seqs using fastq screen 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw
module load bowtie2/2.5.2
module load conda/latest
conda activate fastq_screen
FASTQSCREEN='/home/brooke_sienkiewicz_student_uml_edu/.conda/envs/fastq_screen/share/fastq-screen-0.15.3-0'

spp="PSTR"
SAMPLEFILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/spp_samples"
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
OUTPUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"

gzip *.fastq

while IFS=$'\t' read -r SAMPLEID GROUP; do
     # check if file exists first since first script run failed halfway through
     CHECK_FILE="${OUTPUTDIR}/${SAMPLEID}_host_removed_R2.tagged_filter.fastq.gz"
     if [ -f "$CHECK_FILE" ]; then
         echo "Sample $SAMPLEID already processed. Skipping..."
         continue
     fi
    
     echo "Processing sample: $SAMPLEID"
     $FASTQSCREEN/fastq_screen --nohits --aligner bowtie2 --conf $FASTQSCREEN/fastq_screen.conf --outdir $OUTPUTDIR \
     $READSPATH/"${SAMPLEID}"_host_removed_R1.fastq.gz $READSPATH/"${SAMPLEID}"_host_removed_R2.fastq.gz;

     if [ $? -eq 0 ]; then
             echo "fastq_screen completed successfully for sample: $SAMPLEID"
     else
             echo "fastq_screen encountered an error for sample: $SAMPLEID"
             exit 1
     fi
done < "$SAMPLEFILE"
conda deactivate
echo "Symbiont, human removal: All samples processed successfully."

conda activate assembly 
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/final_filtered"
cd "$READSPATH"

while IFS=$'\t' read -r SAMPLEID GROUP; do
    if [[ -f "${OUTDIR}/${SAMPLEID}_host_removed_R1.tagged_filter_ready.fastq.gz" ]] && [[ -f "${OUTDIR}/${SAMPLEID}_host_removed_R2.tagged_filter_ready.fastq.gz" ]]; then
        echo "${SAMPLEID}: repair already completed"
    else
        repair.sh in1=$READSPATH/"${SAMPLEID}"_host_removed_R1.tagged_filter.fastq.gz in2=$READSPATH/"${SAMPLEID}"_host_removed_R2.tagged_filter.fastq.gz \
        out1=${OUTDIR}/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz out2=${OUTDIR}/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz \
        outs=${OUTDIR}/"${SAMPLEID}"singletons.fq.gz repair tossbrokenreads=t;
         if [ $? -eq 0 ]; then
            echo "repair completed successfully for sample: $SAMPLEID"
        else
            echo "repair encountered an error for sample: $SAMPLEID"
            exit 1
        fi
    fi
done < "$SAMPLEFILE"
echo "Repair: All samples processed successfully."

# 2)concatenate all f and r seqs into single file (1 for f, 1 for r)
    # by group
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
spp_groups=$(cut -f 2 $SAMPLEFILE | sort -u)

for group in $spp_groups; do
    echo "Processing samples for $group"
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    mkdir -p "$OUTDIR"
# make sample list for each group and check if all files exist before concatenating
    samples=$(grep -w "$group" "$SAMPLEFILE" | cut -f 1)
    for s in $samples; do
        [ -f "$READSPATH/${s}_host_removed_R1.tagged_filter_ready.fastq.gz" ] || { echo "Missing R1 for $s"; exit 1; }
        [ -f "$READSPATH/${s}_host_removed_R2.tagged_filter_ready.fastq.gz" ] || { echo "Missing R2 for $s"; exit 1; }
    done

    # concat F & R within group if all files are present 
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R1.tagged_filter_ready.fastq.gz|" | xargs cat > "$OUTDIR/${group}_reads_R1_ALL.fastq.gz"
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R2.tagged_filter_ready.fastq.gz|" | xargs cat > "$OUTDIR/${group}_reads_R2_ALL.fastq.gz"
done

# 3)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, and ensures there are no gaps
for group in $spp_groups; do
    READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    echo "Processing $group"
    MEG_OUT="${READSPATH}/megahit_host_removed"
    rm -rf "$MEG_OUT"

    megahit --presets meta-large \
    -t 20 \
    -1 "${READSPATH}/${group}_reads_R1_ALL.fastq.gz" \
    -2 "${READSPATH}/${group}_reads_R2_ALL.fastq.gz" \
    -o "$MEG_OUT" \
    --out-prefix "$group"
    
    if [ $? -eq 0 ]; then
            echo "megahit completed successfully for $group"
        else
            echo "megahit encountered an error for $group"
            exit 1
        fi
done
# megahit has to make the directory; will fail if it already exists
conda deactivate 

# JOB-ID: 53972008, 54694110, 55241178
# bash script file name: PSTR_assembly2

# JOB-ID: 53972008, 54694110, 55241178, 55305579,55351227, 55386905, 55532725
# bash script file name: PSTR_assembly2

In [ ]:
# pstr issues assembling - tried to rerun a couple of samples that may have given issues in earlier output files 
    # reran fastqscreen and repair for 102019_BEL_CBC_T3_36_PSTR and 102019_BEL_CBC_T1_29_PSTR

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-PSTR_megahit-%j.out  # %j = job ID

# 2)remove symbiont and human seqs using fastq screen 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw
module load bowtie2/2.5.2
module load conda/latest
conda activate fastq_screen
FASTQSCREEN='/home/brooke_sienkiewicz_student_uml_edu/.conda/envs/fastq_screen/share/fastq-screen-0.15.3-0'

spp="PSTR"
SAMPLEFILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/spp_samples"
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
OUTPUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"

#SAMPLEID='102019_BEL_CBC_T3_36_PSTR'
SAMPLEID="102019_BEL_CBC_T1_29_PSTR"

echo "Processing sample: $SAMPLEID"
$FASTQSCREEN/fastq_screen --nohits --aligner bowtie2 --conf $FASTQSCREEN/fastq_screen.conf --outdir $OUTPUTDIR \
$READSPATH/"${SAMPLEID}"_host_removed_R1.fastq.gz $READSPATH/"${SAMPLEID}"_host_removed_R2.fastq.gz;
     if [ $? -eq 0 ]; then
             echo "fastq_screen completed successfully for sample: $SAMPLEID"
     else
             echo "fastq_screen encountered an error for sample: $SAMPLEID"
             exit 1
     fi
conda deactivate
# repair 
conda activate assembly 
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/final_filtered"
cd "$READSPATH"

if [[ -f "${OUTDIR}/${SAMPLEID}_host_removed_R1.tagged_filter_ready.fastq.gz" ]] && [[ -f "${OUTDIR}/${SAMPLEID}_host_removed_R2.tagged_filter_ready.fastq.gz" ]]; then
    echo "${SAMPLEID}: repair already completed"
else
    repair.sh in1=$READSPATH/"${SAMPLEID}"_host_removed_R1.tagged_filter.fastq.gz in2=$READSPATH/"${SAMPLEID}"_host_removed_R2.tagged_filter.fastq.gz \
    out1=${OUTDIR}/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz out2=${OUTDIR}/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz \
    outs=${OUTDIR}/"${SAMPLEID}"singletons.fq.gz repair tossbrokenreads=t;
     if [ $? -eq 0 ]; then
        echo "repair completed successfully for sample: $SAMPLEID"
    else
        echo "repair encountered an error for sample: $SAMPLEID"
        exit 1
        fi
fi

# 2)concatenate all f and r seqs into single file (1 for f, 1 for r)
    # by group
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/final_filtered"
group="102019_PSTR_Healthy"

echo "Processing samples for $group"
OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
# make sample list for each group and check if all files exist before concatenating
samples=$(grep -w "$group" "$SAMPLEFILE" | cut -f 1)
for s in $samples; do
    [ -f "$READSPATH/${s}_host_removed_R1.tagged_filter_ready.fastq.gz" ] || { echo "Missing R1 for $s"; exit 1; }
    [ -f "$READSPATH/${s}_host_removed_R2.tagged_filter_ready.fastq.gz" ] || { echo "Missing R2 for $s"; exit 1; }
done

# concat F & R within group if all files are present 
printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R1.tagged_filter_ready.fastq.gz|" | xargs cat > "$OUTDIR/${group}_reads_R1_ALL.fastq.gz"
printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R2.tagged_filter_ready.fastq.gz|" | xargs cat > "$OUTDIR/${group}_reads_R2_ALL.fastq.gz"


# 3)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, and ensures there are no gaps
spp_groups=$(cut -f 2 $SAMPLEFILE | sort -u)
for group in $spp_groups; do
    READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    echo "Processing $group"
    MEG_OUT="${READSPATH}/megahit_host_removed"
    rm -rf "$MEG_OUT"

    megahit --presets meta-large \
    -t 20 \
    -1 "${READSPATH}/${group}_reads_R1_ALL.fastq.gz" \
    -2 "${READSPATH}/${group}_reads_R2_ALL.fastq.gz" \
    -o "$MEG_OUT" \
    --out-prefix "$group"
    
    if [ $? -eq 0 ]; then
            echo "megahit completed successfully for $group"
        else
            echo "megahit encountered an error for $group"
            exit 1
        fi
done
# megahit has to make the directory; will fail if it already exists
conda deactivate 

# JOB-ID: 55538762, 55648835
# fixed ID 36: 55523611, 55509313
# bash script file name: PSTR_singlesample_megahit

In [ ]:
# pstr still giving issues so redid individually
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=250G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long
#SBATCH -t 96:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH --mail-type=TIME_LIMIT_80
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-PSTR_assemblyv2-%j.out  # %j = job ID

# 2)remove symbiont and human seqs using fastq screen 
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw
module load bowtie2/2.5.2
module load conda/latest
conda activate fastq_screen
FASTQSCREEN='/home/brooke_sienkiewicz_student_uml_edu/.conda/envs/fastq_screen/share/fastq-screen-0.15.3-0'

spp="PSTR"
SAMPLEFILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/spp_samples"
# READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
# OUTPUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly2/sym_human_removed"

# mkdir -p "$OUTPUTDIR"

# while IFS=$'\t' read -r SAMPLEID GROUP; do
#      echo "Processing sample: $SAMPLEID"
#      $FASTQSCREEN/fastq_screen --nohits --aligner bowtie2 --conf $FASTQSCREEN/fastq_screen.conf --outdir $OUTPUTDIR \
#      $READSPATH/"${SAMPLEID}"_host_removed_R1.fastq.gz $READSPATH/"${SAMPLEID}"_host_removed_R2.fastq.gz;

#      if [ $? -eq 0 ]; then
#              echo "fastq_screen completed successfully for sample: $SAMPLEID"
#      else
#              echo "fastq_screen encountered an error for sample: $SAMPLEID"
#              exit 1
#      fi
# done < "$SAMPLEFILE"
# conda deactivate
# echo "Symbiont, human removal: All samples processed successfully."

# conda activate assembly 
# READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly2/sym_human_removed"
# OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly2/final_filtered"
# cd "$READSPATH"
# mkdir -p "$OUTDIR"

# while IFS=$'\t' read -r SAMPLEID GROUP; do
#     repair.sh in1=$READSPATH/"${SAMPLEID}"_host_removed_R1.tagged_filter.fastq.gz in2=$READSPATH/"${SAMPLEID}"_host_removed_R2.tagged_filter.fastq.gz \
#     out1=${OUTDIR}/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz out2=${OUTDIR}/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz \
#     outs=${OUTDIR}/"${SAMPLEID}"singletons.fq.gz repair tossbrokenreads=t;
#     if [ $? -eq 0 ]; then
#         echo "repair completed successfully for sample: $SAMPLEID"
#     else
#         echo "repair encountered an error for sample: $SAMPLEID"
#         exit 1
#     fi
# done < "$SAMPLEFILE"
# echo "Repair: All samples processed successfully."

# 2)concatenate all f and r seqs into single file (1 for f, 1 for r)
    # by group
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly2/final_filtered"
spp_groups=$(cut -f 2 $SAMPLEFILE | sort -u)

for group in $spp_groups; do
    echo "Processing samples for $group"
    OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly2/${group}"
    mkdir -p "$OUTDIR"
# make sample list for each group and check if all files exist before concatenating
    samples=$(grep -w "$group" "$SAMPLEFILE" | cut -f 1)
    for s in $samples; do
        [ -f "$READSPATH/${s}_host_removed_R1.tagged_filter_ready.fastq.gz" ] || { echo "Missing R1 for $s"; exit 1; }
        [ -f "$READSPATH/${s}_host_removed_R2.tagged_filter_ready.fastq.gz" ] || { echo "Missing R2 for $s"; exit 1; }
    done

    # concat F & R within group if all files are present 
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R1.tagged_filter_ready.fastq.gz|" | xargs cat > "$OUTDIR/${group}_reads_R1_ALL.fastq.gz"
    printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R2.tagged_filter_ready.fastq.gz|" | xargs cat > "$OUTDIR/${group}_reads_R2_ALL.fastq.gz"
done

# 3)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, and ensures there are no gaps
for group in $spp_groups; do
    READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly2/${group}"
    echo "Processing $group"
    MEG_OUT="${READSPATH}/megahit_host_removed"
    rm -rf "$MEG_OUT"

    megahit --presets meta-large \
    -t 20 \
    -1 "${READSPATH}/${group}_reads_R1_ALL.fastq.gz" \
    -2 "${READSPATH}/${group}_reads_R2_ALL.fastq.gz" \
    -o "$MEG_OUT" \
    --out-prefix "$group"
    
    if [ $? -eq 0 ]; then
            echo "megahit completed successfully for $group"
        else
            echo "megahit encountered an error for $group"
            exit 1
        fi
done
# megahit has to make the directory; will fail if it already exists
conda deactivate 

# JOB-ID: 60621655 - bad path for input of megahit. successfully ran through repair
# bash script file name: PSTR_assemblyv2

In [ ]:
#### redo pstr megahit 

In [ ]:
#!/bin/bash
#SBATCH -c 20  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --array=1-7
#SBATCH --mail-type=ALL --mail-type=TIME_LIMIT_80
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/PSTR/slurm-PSTR_megahit-%A_%a.out  # %j = job ID

# redo pstr megahit
# array jobs for each group
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw
module load conda/latest
conda activate assembly

spp="PSTR"
# get group list for pstr - use as array job input 
SAMPLE_FILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/filtered_sample_groups.txt"
group=$(grep -w $spp $SAMPLE_FILE | cut -f 3 | sort | uniq |sed -n "${SLURM_ARRAY_TASK_ID}p") 

# 3)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, and ensures there are no gaps
echo "Processing $group"
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
MEG_OUT="${READSPATH}/megahit_host_removed"
rm -rf "$MEG_OUT"

megahit --presets meta-large \
        -t 20 \
        -1 "${READSPATH}/${group}_reads_R1_ALL.fastq.gz" \
        -2 "${READSPATH}/${group}_reads_R2_ALL.fastq.gz" \
        -o "$MEG_OUT" \
        --out-prefix "$group" # megahit has to make the directory; will fail if it already exists
if [ $? -eq 0 ]; then
        echo "megahit completed successfully for $group"
    else
        echo "megahit encountered an error for $group"
        exit 1
    fi

conda deactivate 
# JOB-ID: 
# bash script file name: PSTR_megahit

In [ ]:
# negatives 
    # start with host removal - all references i guess?
    # symbiont, human removal
    # dont need to assemble with megahit

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=250G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long
#SBATCH -t 144:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/slurm-NEG_assembly2-%j.out  # %j = job ID

# filter for coral host genomes

# set paths for existing bowtie genome indices - (all genomes we have)
MCAV_index=Mcav_DB
MCAV_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Mcav_genome/"
MMEA_index=Mmea_DB
MMEA_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Mmea_genome/"
ORBI_index=Ofav_DB
ORBI_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Ofav_genome/"
SSID_index=Ssid_DB
SSID_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Ssid_genome/"
CNAT_index=Cnat_DB
CNAT_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Cnat_genome/"

# concat these into a list
spp_list=("MCAV" "MMEA" "ORBI" "SSID" "CNAT")

# samplelist
spp="NEG"
SAMPLEFILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/spp_samples"
READSPATH="/scratch/workspace/brooke_sienkiewicz_student_uml_edu-raw_seqs/trimmed"
WORKINGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed/temp"        
FINALREADS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed"
mkdir -p $WORKINGPATH
mkdir -p $FINALREADS

# align negative seqs to host genomes
while IFS= read -r id; do
    echo "Processing sample: ${id}"
    CURRENT_R1="$READSPATH/${id}_R1_001_val_1.fq.gz"
    CURRENT_R2="$READSPATH/${id}_R2_001_val_2.fq.gz"
#loop through list of genomes
    for x in "${spp_list[@]}"; do
        # assigning path and index variable for each host genome and run alignment
        idx_var="${x}_index"
        path_var="${x}_path"
        input_index="${!idx_var}"
        input_path="${!path_var}"
        
        bowtie2 -p 8 --very-sensitive \
            -x "$input_path/$input_index" \
            -1 "$CURRENT_R1" \
            -2 "$CURRENT_R2" \
            -S /dev/null \
            --un-conc "$WORKINGPATH/${id}_unmapped_${x}_R%.fastq" \
            2> "$FINALREADS/${id}_${x}_alignment_report.txt"
        if [ $? -eq 0 ]; then
            echo "${x} removal completed successfully for sample: ${id}"
        else
            echo "${x} removal encountered an error for sample: ${id}" 
        fi
        
        # update input read for next genome..
        CURRENT_R1="$WORKINGPATH/${id}_unmapped_${x}_R1.fastq"
        CURRENT_R2="$WORKINGPATH/${id}_unmapped_${x}_R2.fastq"
    done
# Move the final filtered files to the results folder
    mv "$CURRENT_R1" "$FINALREADS/${id}_final_host_removed_R1.fastq"
    mv "$CURRENT_R2" "$FINALREADS/${id}_final_host_removed_R2.fastq"
# Clean up the intermediate temp files for this sample
    rm $WORKINGPATH/${id}_unmapped_*
done < "$SAMPLEFILE"

conda deactivate
gzip "$FINALREADS"/*.fastq
echo "Host removal: All samples processed successfully."


# 2)remove symbiont and human seqs using fastq screen 
module load bowtie2/2.5.2
module load conda/latest
conda activate fastq_screen
FASTQSCREEN='/home/brooke_sienkiewicz_student_uml_edu/.conda/envs/fastq_screen/share/fastq-screen-0.15.3-0'
READSPATH='/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/host_removed'
OUTPUTDIR='/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed'

while IFS= read -r SAMPLEID; do
    # check if file exists first since first script run failed halfway through
    CHECK_FILE="${OUTPUTDIR}/${SAMPLEID}_host_removed_R1.tagged_filter.fastq.gz"
    if [ -f "$CHECK_FILE" ]; then
        echo "Sample $SAMPLEID already processed. Skipping..."
        continue
    fi
    
    echo "Processing sample: $SAMPLEID"
    $FASTQSCREEN/fastq_screen --nohits --aligner bowtie2 --conf $FASTQSCREEN/fastq_screen.conf --outdir $OUTPUTDIR \
    $READSPATH/"${SAMPLEID}"_host_removed_R1.fastq.gz $READSPATH/"${SAMPLEID}"_host_removed_R2.fastq.gz;
    if [ $? -eq 0 ]; then
            echo "fastq_screen completed successfully for sample: $SAMPLEID"
    else
            echo "fastq_screen encountered an error for sample: $SAMPLEID"
            exit 1
    fi
done < "$SAMPLEFILE"
conda deactivate
echo "Symbiont, human removal: All samples processed successfully."

conda activate assembly 
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/final_filtered"
mkdir -p "$OUTDIR"

while IFS= read -r SAMPLEID; do
    repair.sh in1=$READSPATH/"${SAMPLEID}"_host_removed_R1.tagged_filter.fastq.gz in2=$READSPATH/"${SAMPLEID}"_host_removed_R2.tagged_filter.fastq.gz \
    out1=${OUTDIR}/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz out2=${OUTDIR}/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz \
    outs=${OUTDIR}/"${SAMPLEID}"singletons.fq repair;
     if [ $? -eq 0 ]; then
        echo "repair completed successfully for sample: $SAMPLEID"
    else
        echo "repair encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$SAMPLEFILE"
echo "Repair: All samples processed successfully."
conda deactivate

# JOB-ID: 55308068, 55387834,55351376, 55328337
# bash script file name: neg_assembly2

In [ ]:
# check output of negatives
grep -H "overall alignment rate" $FINALREADS/*_alignment_report.txt

In [ ]:
# below is the fastqscreen.conf file 

In [ ]:
############################
## Bowtie, Bowtie 2 or BWA #
############################
## If the Bowtie, Bowtie 2 or BWA binary is not in your PATH, you can set 
## this value to tell the program where to find your chosen aligner.  Uncomment 
## the relevant line below and set the appropriate location.  Please note, 
## this path should INCLUDE the executable filename.

#BOWTIE	/usr/local/bin/bowtie/bowtie
#BOWTIE2 /usr/local/bowtie2/bowtie2
#BWA /usr/local/bwa/bwa

############################################
## Bismark (for bisulfite sequencing only) #
############################################
## If the Bismark binary is not in your PATH then you can set this value to 
## tell the program where to find it.  Uncomment the line below and set the 
## appropriate location. Please note, this path should INCLUDE the executable 
## filename.

#BISMARK	/usr/local/bin/bismark/bismark

############
## Threads #
############
## Genome aligners can be made to run across multiple CPU cores to speed up 
## searches.  Set this value to the number of cores you want for mapping reads.

THREADS		12

##############
## DATABASES #
##############
## This section enables you to configure multiple genomes databases (aligner index 
## files) to search against in your screen.  For each genome you need to provide a 
## database name (which can't contain spaces) and the location of the aligner index 
## files.
##
## The path to the index files SHOULD INCLUDE THE BASENAME of the index, e.g:
## /data/public/Genomes/Human_Bowtie/GRCh37/Homo_sapiens.GRCh37
## Thus, the index files (Homo_sapiens.GRCh37.1.bt2, Homo_sapiens.GRCh37.2.bt2, etc.) 
## are found in a folder named 'GRCh37'.
##
## If, for example, the Bowtie, Bowtie2 and BWA indices of a given genome reside in 
## the SAME FOLDER, a SINLGE path may be provided to ALL the of indices.  The index 
## used will be the one compatible with the chosen aligner (as specified using the 
## --aligner flag).  
##
## The entries shown below are only suggested examples, you can add as many DATABASE 
## sections as required, and you can comment out or remove as many of the existing 
## entries as desired.  We suggest including genomes and sequences that may be sources 
## of contamination either because they where run on your sequencer previously, or may 
## have contaminated your sample during the library preparation step.
##
## Human - sequences available from
## ftp://ftp.ensembl.org/pub/current/fasta/homo_sapiens/dna/
## (Kraken2 RefSeq db)
DATABASE	Human	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/ref_databases/standard/library/human/index
##
## added more databases and updated a few listed here with their updated assemblies 12.11.2024
## Symbionts
DATABASE	Symbiont1	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/Durusdinium_trenchii_indexed
## Symbionts
DATABASE	Symbiont2	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_000507305.1_index
## Symbionts
DATABASE	Symbiont3	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_001939145.1_index
## Symbionts
DATABASE	Symbiont4	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_003297005.1_index
## Symbionts
DATABASE	Symbiont5	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_009767595.1_index
## Symbionts
DATABASE	Symbiont6	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_018327485.1_index
## Symbionts
DATABASE	Symbiont7	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_905221635.1_index
## Symbionts
DATABASE	Symbiont8	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_947184155.2_index
## Symbionts
DATABASE	Symbiont9	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_003297045.1_index
## Symbionts
DATABASE	Symbiont10	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_905231905.1_index
## Symbionts
DATABASE	Symbiont11	/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/symbionts/indexed/GCA_905231915.1_index
##
## Ecoli- sequence available from EMBL accession U00096.2
#DATABASE	Ecoli	/data/public/Genomes/Ecoli/Ecoli
##
## PhiX - sequence available from Refseq accession NC_001422.1
#DATABASE	PhiX	/data/public/Genomes/PhiX/phi_plus_SNPs
##
## Adapters - sequence derived from the FastQC contaminats file found at: www.bioinformatics.babraham.ac.uk/projects/fastqc
#DATABASE	Adapters	/data/public/Genomes/Contaminants/Contaminants
##
## Vector - Sequence taken from the UniVec database
## http://www.ncbi.nlm.nih.gov/VecScreen/UniVec.html
#DATABASE	Vectors		/data/public/Genomes/Vectors/Vectors

In [ ]:
# make script to zip all intermediate files
# SPP= 
# OUTPUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
# TAGGEDDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed/symb_seqs"

# # go ahead and zip tagged.fastq files 
# mkdir -p "$TAGGEDDIR"
# cd "$OUTPUTDIR"

# rsync -av "$OUTPUTDIR" "$TAGGEDDIR"
# ZSTD_NBTHREADS=0 tar --zstd -cf raw.tar.zst raw

# if [ $? -eq 0 ]; then
#    echo "Zipped "raw" successfully!"
# else
#    echo "Command failed! (zipping "raw")"
# fi
# # test integrity of zipped dir
# zstd -t raw.tar.zst
# if [ $? -eq 0 ]; then
#    echo "Integrity check passed! Deleting uncompressed copy at destination..."
#    rm -rf /scratch/workspace/brooke_sienkiewicz_student_uml_edu-raw_seqs/raw
# else
#    echo "CRITICAL ERROR: Integrity check failed. Keeping uncompressed 'raw' for safety."
#    exit 1
# fi

### QA/QC

In [ ]:
# manually run one truncated PSTR sample
salloc --cpus-per-task=16 --mem=62G --time=04:00:00

spp="PSTR"
id="102019_BEL_CBC_T1_29_PSTR"
READSPATH="/scratch/workspace/brooke_sienkiewicz_student_uml_edu-raw_seqs/trimmed"
WORKINGPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly_old/host_removed/temp"
FINALREADS="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly_old/host_removed"

PSTR_index=Cnat_DB
PSTR_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Cnat_genome/"
     
input_index="Cnat_DB"
input_path="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Cnat_genome"


module load conda/latest
conda activate anvio-8

# running 1 at a time - currently on bowtie
bowtie2 -p 16 -x $input_path/$input_index \
    -1 "$READSPATH/${id}_R1_001_val_1.fq.gz" \
    -2 "$READSPATH/${id}_R2_001_val_2.fq.gz" | \
    samtools view -@ 6 -b -f 12 -F 256 - > "$WORKINGPATH/${id}_bothReadsUnmapped.bam"
        
# sorts the file so both mates are together and then extracts them back as .fastq files
samtools sort -n -m 4G -@ 12 "$WORKINGPATH/${id}_bothReadsUnmapped.bam" -o "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam"
samtools fastq -@ 16 "$WORKINGPATH/${id}_bothReadsUnmapped_sorted.bam" \
    -1 >(bgzip -c > "$FINALREADS/${id}_host_removed_R1.fastq.gz") \
    -2 >(bgzip -c > "$FINALREADS/${id}_host_removed_R2.fastq.gz") \
    -0 /dev/null -s /dev/null -n

# delete intermediate bam files
rm -f "$WORKINGPATH/${id}"_*.bam

# Test if the gzip archive is complete and untruncated
gzip -t "$FINALREADS/${id}_host_removed_R1.fastq.gz" && echo "File is healthy!"

# Run FastQC on the single file
fastqc -t 4 -o "$FINALREADS" "$FINALREADS/${id}_host_removed_R1.fastq.gz"
fastqc -t 4 -o "$FINALREADS" "$FINALREADS/${id}_host_removed_R2.fastq.gz"

# rerun symbiont removal
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw
module load bowtie2/2.5.2
module load conda/latest
conda activate fastq_screen
FASTQSCREEN='/home/brooke_sienkiewicz_student_uml_edu/.conda/envs/fastq_screen/share/fastq-screen-0.15.3-0'

spp="PSTR"
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly_old/host_removed"
OUTPUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
SAMPLEID="102019_BEL_CBC_T1_29_PSTR"

ls "$OUTPUTDIR/${SAMPLEID}"_host_removed*
mv "$OUTPUTDIR/${SAMPLEID}"_host_removed* "$OUTPUTDIR/temp/${SAMPLEID}"_host_removed*

$FASTQSCREEN/fastq_screen --nohits --aligner bowtie2 --conf $FASTQSCREEN/fastq_screen.conf --outdir $OUTPUTDIR \
$READSPATH/"${SAMPLEID}"_host_removed_R1.fastq.gz $READSPATH/"${SAMPLEID}"_host_removed_R2.fastq.gz

conda deactivate


# repair
spp="PSTR"
SAMPLEID="102019_BEL_CBC_T1_29_PSTR"
module load conda/latest
conda activate assembly

READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/sym_human_removed"
OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/final_filtered"
cd "$OUTDIR"

repair.sh in1=$READSPATH/"${SAMPLEID}"_host_removed_R1.tagged_filter.fastq.gz in2=$READSPATH/"${SAMPLEID}"_host_removed_R2.tagged_filter.fastq.gz \
out1=${OUTDIR}/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz out2=${OUTDIR}/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz \
outs=${OUTDIR}/"${SAMPLEID}"singletons.fq repair;
     if [ $? -eq 0 ]; then
        echo "repair completed successfully for sample: $SAMPLEID"
    else
        echo "repair encountered an error for sample: $SAMPLEID"
        exit 1
    fi

# 2) re-concatenate all f and r seqs into single file (1 for f, 1 for r)
SAMPLEFILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/spp_samples"
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/final_filtered"
group=$(tail -n +2 "$SAMPLEFILE" | grep -w ${SAMPLEID} | cut -f 2 | uniq)

OUTDIR="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
rm -rf "$OUTDIR"    # delete and redo 
mkdir -p "$OUTDIR"
# make sample list for each group and check if all files exist before concatenating
samples=$(grep -w "$group" "$SAMPLEFILE" | cut -f 1)
for s in $samples; do
    [ -f "$READSPATH/${s}_host_removed_R1.tagged_filter_ready.fastq.gz" ] || { echo "Missing R1 for $s"; exit 1; }
    [ -f "$READSPATH/${s}_host_removed_R2.tagged_filter_ready.fastq.gz" ] || { echo "Missing R2 for $s"; exit 1; }
done

# concat F & R within group if all files are present 
printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R1.tagged_filter_ready.fastq.gz|" | xargs cat > "$OUTDIR/${group}_reads_R1_ALL.fastq.gz"
printf "%s\n" $samples | sed "s|^|$READSPATH/|; s|$|_host_removed_R2.tagged_filter_ready.fastq.gz|" | xargs cat > "$OUTDIR/${group}_reads_R2_ALL.fastq.gz"

In [ ]:
# redo pstr in sbatch

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 15:00:00  # Job time limit
#SBATCH --mail-type=ALL,TIME_LIMIT_80
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/PSTR/slurm-PSTR_singlegroup_megahit-%j.out

spp="PSTR"
SAMPLEFILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/spp_samples"
group=$(tail -n +2 "$SAMPLEFILE" | grep -w ${SAMPLEID} | cut -f 2 | uniq)

module load conda/latest
conda activate assembly

# 3) redo single group assembly 
READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
echo "Processing $group"
MEG_OUT="${READSPATH}/megahit_host_removed"
rm -rf "$MEG_OUT"

megahit --presets meta-large \
    -t 24 \
    -1 "${READSPATH}/${group}_reads_R1_ALL.fastq.gz" \
    -2 "${READSPATH}/${group}_reads_R2_ALL.fastq.gz" \
    -o "$MEG_OUT" \
    --out-prefix "$group"
    
if [ $? -eq 0 ]; then
        echo "megahit completed successfully for $group"
    else
        echo "megahit encountered an error for $group"
        exit 1
    fi
    
conda deactivate

# job ID: 61821010
# job file: pstr_singlegroup_megahit

In [ ]:
### found issues in MCAV assemblies ###

In [ ]:
###### check all the group assemblies 

In [ ]:
SAMPLE_FILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/filtered_sample_groups.txt"
groups=$(tail -n +2 "$SAMPLE_FILE" | cut -f 3 |sort| uniq)

for group in $groups; do
    spp=$(grep -P "\t${group}$" "$SAMPLE_FILE" | cut -f 2 | head -n 1)
    file_path="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}/${group}_reads_R2_ALL.fastq.gz"

    if [[ -f "$file_path" ]]; then

        file_type=$(zcat "$file_path" | file - | cut -d: -f2)
        echo "Group: ${group} --> Content:${file_type}"
    else
        echo "Group: ${group} --> File missing!"
    fi
done

In [ ]:
Group: 102019_PSTR_Healthy --> Content: ASCII text
Group: 122022_MCAV_Diseased_Margin --> Content: gzip compressed data, from FAT filesystem (MS-DOS, OS/2, NT)
Group: 122022_MCAV_Diseased_Tissue --> Content: gzip compressed data, from FAT filesystem (MS-DOS, OS/2, NT)
Group: 122022_MCAV_Healthy --> Content: gzip compressed data, from FAT filesystem (MS-DOS, OS/2, NT)
Group: 122022_OANN_Diseased_Margin --> Content: ASCII text
Group: 122022_OANN_Diseased_Tissue --> Content: ASCII text
Group: 122022_OANN_Healthy --> Content: ASCII text
Group: 122022_OFAV_Diseased_Margin --> Content: ASCII text
Group: 122022_OFAV_Diseased_Tissue --> Content: ASCII text
Group: 122022_OFAV_Healthy --> Content: ASCII text
Group: 122022_PAST_Diseased_Margin --> Content: ASCII text
Group: 122022_PAST_Diseased_Tissue --> Content: ASCII text
Group: 122022_PAST_Healthy --> Content: ASCII text
Group: 122022_PSTR_Diseased_Margin --> Content: ASCII text
Group: 122022_PSTR_Diseased_Tissue --> Content: ASCII text
Group: 122022_PSTR_Healthy --> Content: ASCII text
Group: 52022_MCAV_Diseased_Margin --> Content: gzip compressed data, from FAT filesystem (MS-DOS, OS/2, NT)
Group: 52022_MCAV_Diseased_Tissue --> Content: gzip compressed data, from FAT filesystem (MS-DOS, OS/2, NT)
Group: 52022_MCAV_Healthy --> Content: gzip compressed data, from FAT filesystem (MS-DOS, OS/2, NT)
Group: 52022_OANN_Healthy --> Content: ASCII text
Group: 52022_OFAV_Healthy --> Content: ASCII text
Group: 52022_PAST_Diseased_Margin --> Content: ASCII text
Group: 52022_PAST_Diseased_Tissue --> Content: ASCII text
Group: 52022_PAST_Healthy --> Content: ASCII text
Group: 52022_PSTR_Diseased_Margin --> Content: ASCII text
Group: 52022_PSTR_Diseased_Tissue --> Content: ASCII text
Group: 52022_PSTR_Healthy --> Content: ASCII text
Group: 62019_MCAV_Healthy --> Content: gzip compressed data, from FAT filesystem (MS-DOS, OS/2, NT)
Group: 62019_MMEA_Healthy --> Content: ASCII text
Group: 62019_PAST_Healthy --> Content: ASCII text
Group: Negative --> File missing!

In [ ]:
# ok great its just the MCAVs - so redo mcav megahit only
# unzip the double zipped files 
spp="MCAV"
groups=$(tail -n +2 "$SAMPLE_FILE" | grep -w "$spp" | cut -f 3 | sort -u)

for group in $groups; do
    file_path="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    
    R1="${group}_reads_R1_ALL.fastq.gz"
    R2="${group}_reads_R2_ALL.fastq.gz"

    if [[ -f "$file_path/$R1" ]]; then
        zcat "$file_path/$R1" > "$file_path/${group}_reads_R1_ALL_temp.fastq.gz" \
          && mv "$file_path/${group}_reads_R1_ALL_temp.fastq.gz" "$file_path/$R1"
    fi

    if [[ -f "$file_path/$R2" ]]; then
        zcat "$file_path/$R2" > "$file_path/${group}_reads_R2_ALL_temp.fastq.gz" \
          && mv "$file_path/${group}_reads_R2_ALL_temp.fastq.gz" "$file_path/$R2"
    fi
done


# script improved by gemini: 
for group in $groups; do
    echo "============================================"
    echo "Processing Group: $group"
    echo "============================================"
    
    file_path="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
    
    for r in R1 R2; do
        filename="${group}_reads_${r}_ALL.fastq.gz"
        full_target="$file_path/$filename"
        temp_target="$file_path/${group}_reads_${r}_ALL_temp.fastq.gz"
        
        if [[ -f "$full_target" ]]; then
            # Check file status before fix
            before_type=$(zcat "$full_target" 2>/dev/null | file - | cut -d: -f2 | xargs)
            echo "[$r BEFORE] Content type: $before_type"
            
            echo "--> Stripping outer gzip layer from $filename..."
            
            if zcat "$full_target" > "$temp_target" && mv "$temp_target" "$full_target"; then
                # Check file status after fix
                after_type=$(zcat "$full_target" 2>/dev/null | file - | cut -d: -f2 | xargs)
                echo "[$r AFTER ] Content type: $after_type"
                echo "[SUCCESS] Successfully updated $filename"
            else
                echo "[ERROR] Failed to process $filename"
            fi
        else
            echo "[SKIP] File not found: $full_target"
        fi
        echo "--------------------------------------------"
    done

# confirmed fixed!!
# just need to rerun megahit on MCAV now 

In [ ]:
#### redo MCAV megahit assemblies bc concatenated reads were double zipped #####

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=62G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --array=1-7
#SBATCH --mail-type=ALL,TIME_LIMIT_80
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs042024/outputs/MCAV/slurm-mcav_megahit_redo-%A-%a.out  # %j = job ID

module load conda/latest
conda activate assembly

SAMPLE_FILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/filtered_sample_groups.txt"
group=$(tail -n +2 "$SAMPLE_FILE" | cut -f 3 | grep "MCAV" | sort | uniq | sed -n "${SLURM_ARRAY_TASK_ID}p")

# get group and species from sampleid
spp="MCAV"

READSPATH="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/${spp}/assembly/${group}"
echo "Processing $group"
MEG_OUT="${READSPATH}/megahit_host_removed"
rm -rf "$MEG_OUT"

megahit --presets meta-large \
    -t 24 \
    -1 "${READSPATH}/${group}_reads_R1_ALL.fastq.gz" \
    -2 "${READSPATH}/${group}_reads_R2_ALL.fastq.gz" \
    -o "$MEG_OUT" \
    --out-prefix "$group"
    
if [ $? -eq 0 ]; then
        echo "megahit completed successfully for $group"
    else
        echo "megahit encountered an error for $group"
        exit 1
    fi
conda deactivate

# job ID: 61822228
# job file: mcav_megahit_redo